In [19]:
import os

# Hard cap threads across common native libs (must be set before numpy/scipy import)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ.setdefault("MallocNanoZone", "0")

# If you see OpenMP duplicate runtime issues on mac, this can prevent aborts
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")


'TRUE'

In [20]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.linalg import lstsq
from sklearn.model_selection import train_test_split
import cvxpy as cp
from itertools import product

import matplotlib
matplotlib.use("Agg")  # disables interactive macOS GPU-backed rendering
import matplotlib.pyplot as plt
plt.ioff()


In [21]:
import psutil, gc, time
proc = psutil.Process(os.getpid())

def rss_gb():
    return proc.memory_info().rss / (1024**3)

def log_mem(tag):
    print(f"[mem] {tag:30s} RSS={rss_gb():.2f} GB")


RSS0 = None
def mem_alarm(tag="", growth_gb=1.0):
    global RSS0
    r = rss_gb()
    if RSS0 is None:
        RSS0 = r
    if (r - RSS0) > growth_gb:
        print(f"[mem-alarm] {tag} grew by {r - RSS0:.2f} GB (RSS0={RSS0:.2f} -> {r:.2f})")
        return True
    return False


In [22]:
import time, gc

def reset_inner_solver(inner, N, q_max, delta_t, tag=""):
    """Hard-reset CVXPY/GUROBI-related objects to reduce long-run memory growth."""
    try:
        del inner
    except Exception:
        pass

    gc.collect()
    time.sleep(5)  # small pause helps macOS reclaim/rebalance

    inner = InnerSolver(N=N, q_max=q_max, delta_t=delta_t)

    log_mem(f"reset_inner_solver {tag}")
    return inner


In [23]:
def safe_solve(inner, *, C, q_hp, delta_T_i, delta_T_a, q_solar, Q_sc, weights,
               solver=cp.GUROBI, max_retries=1, tag=""):
    for attempt in range(max_retries + 1):
        try:
            out = inner.solve(
                C=C,
                q_hp=q_hp,
                delta_T_i=delta_T_i,
                delta_T_a=delta_T_a,
                q_solar=q_solar,
                Q_sc=Q_sc,
                weights=weights,
                solver=solver
            )
            return out, inner

        except Exception as e:
            print(f"[solve error] {tag} C={C} weights={weights} attempt={attempt+1}")
            print("Exception:", repr(e))
            log_mem("at solve exception")

            # hard reset and retry once
            inner = reset_inner_solver(
                inner, N=inner.N, q_max=inner.q_max, delta_t=inner.delta_t,
                tag=f"retry after exception {tag}"
            )

            if attempt == max_retries:
                return None, inner

    return None, inner


In [24]:
class InnerSolver:
    def __init__(self, N, q_max, delta_t):
        self.N = int(N)
        self.delta_t = float(delta_t)
        self.q_max = float(q_max)

        # Parameters (data that changes per call)
        #self.C      = cp.Parameter(nonneg=True)         # scalar
        self.invC   = cp.Parameter(nonneg=True)         # scalar
        self.ph_q   = cp.Parameter(nonneg=True)         # scalar
        self.phi_e  = cp.Parameter(nonneg=True)         # scalar
        self.phi_u  = cp.Parameter(nonneg=True)         # scalar

        self.q_hp       = cp.Parameter(self.N)          # vector
        self.delta_T_i  = cp.Parameter(self.N)          # vector
        #self.delta_T_a  = cp.Parameter(self.N)          # vector
        #self.q_solar    = cp.Parameter(self.N)          # vector

        self.Q_sc       = cp.Parameter()                # scalar

        self.X_a = cp.Parameter(self.N)   # equals delta_T_a * delta_t (numpy)
        self.X_s = cp.Parameter(self.N)   # equals q_solar (numpy)

        # Variables
        self.a    = cp.Variable(pos=True)
        self.w_s  = cp.Variable(nonneg=True)
        self.w    = cp.Variable()
        self.e    = cp.Variable()
        self.z    = cp.Variable(self.N, nonneg=True)    # currently unused
        self.u    = cp.Variable(self.N)
        self.q_hat = cp.Variable(self.N)

        # Constant vector of ones (avoid re-allocations)
        self.ones = np.ones(self.N)

        # Model expression
        expr = (self.a * self.X_a
                + self.q_hat
                + self.w_s * self.X_s
                + self.w * self.ones * self.delta_t)


        constraints = [
            self.e == self.Q_sc - cp.sum(self.q_hat),
            #self.delta_T_i + self.u == expr / self.C,
            self.delta_T_i + self.u == cp.multiply(expr, self.invC),
            self.q_hat >= 0,
            self.q_hat <= q_max,
            self.a >= 1/25
        ]

        # Objective (same structure as yours)
        obj = cp.Minimize(
            self.ph_q  * cp.norm1(self.q_hat - self.q_hp) * self.delta_t
            + self.phi_e * cp.square(self.e) * self.delta_t
            + self.phi_u * cp.norm2(self.u) * self.delta_t
        )

        self.prob = cp.Problem(obj, constraints)
        print("DCP:", self.prob.is_dcp(), "DPP:", self.prob.is_dpp())


    def solve(self, C, q_hp, delta_T_i, delta_T_a, q_solar, Q_sc, weights,
              solver=cp.GUROBI):
        # Set parameter values (no new graph/model build)
        #self.C.value = float(C)
        C = float(C)
        self.invC.value = 1.0 / C
        self.q_hp.value = np.asarray(q_hp, dtype=float)
        self.delta_T_i.value = np.asarray(delta_T_i, dtype=float)
        #self.delta_T_a.value = np.asarray(delta_T_a, dtype=float)
        #self.q_solar.value = np.asarray(q_solar, dtype=float)
        self.Q_sc.value = float(Q_sc)

        self.X_a.value = np.asarray(delta_T_a, dtype=float) * self.delta_t
        self.X_s.value = np.asarray(q_solar, dtype=float)

        self.ph_q.value  = float(weights[0])
        self.phi_e.value = float(weights[1])
        self.phi_u.value = float(weights[2])


        self.prob.solve(solver=solver, warm_start=True, verbose=False,
                        Threads = 1)

        # -------------------------
        # GUARD: solver failures => variable values can be None
        # -------------------------
        if self.prob.status not in ("optimal", "optimal_inaccurate"):
            raise RuntimeError(f"CVXPY status={self.prob.status}")

        if (self.a.value is None) or (self.w_s.value is None) or (self.w.value is None) \
           or (self.e.value is None) or (self.u.value is None) or (self.q_hat.value is None):
            raise RuntimeError("CVXPY returned None variable values (likely failed solve).")

        # Extract values
        a_val = float(self.a.value)
        R_a = 1.0 / a_val
        ws_val = float(self.w_s.value)
        w_val = float(self.w.value)
        q_hat_val = np.asarray(self.q_hat.value).copy()  # copy so caller can keep it safely



        # Metrics (fix scaling)
        # u is in temperature-units per step (since delta_T_i + u = ...)
        rmse_dTi = float(np.sqrt(np.mean(np.square(self.u.value))))

        # q_hat and q_hp are in same units (whatever you pass in); don't multiply one side by delta_t
        rmse_q = float(np.sqrt(np.mean(np.square(self.q_hat.value - self.q_hp.value))))

        cost_q = float(np.linalg.norm(self.q_hat.value - self.q_hp.value, 1) * self.delta_t)
        cost_e = float((self.e.value ** 2) * self.delta_t)
        cost_u = float(np.linalg.norm(self.u.value, 2) * self.delta_t)

        return R_a, ws_val, w_val, rmse_q, rmse_dTi, q_hat_val, cost_q, cost_e, cost_u


In [25]:
#1 norm no q_hp upper constraint at each timestep
def solve_inner(C, q_hp, delta_T_i, delta_T_a, q_solar, SPC_sum, DHW_sum,
                Q_DHW_estimate, q_max, delta_t, weights):
    """Solve inner CVXPY problem for a given C and return parameters + metrics."""
    N = len(delta_T_i)
    Q_sc = np.sum(q_hp) - Q_DHW_estimate + DHW_sum

    # CVXPY variables
    a = cp.Variable(pos=True)
    w_s = cp.Variable(nonneg=True)
    #w_w = cp.Variable(nonneg=True)
    w = cp.Variable()
    e = cp.Variable()
    z = cp.Variable(N, nonneg=True)
    u = cp.Variable(N)
    q_hat = cp.Variable(N)

    # Expression
    expr = (delta_T_a * a * delta_t + q_hat + w_s * q_solar + w * np.ones(N) *
            delta_t)

    # Constraints
    constraints = [
        e == Q_sc - cp.sum(q_hat),
        delta_T_i + u == expr / C,
        q_hat >= 0,
        q_hat <= q_max,
        a >= 1 / 25
    ]
            #q_hat <= q_hp + z,


    # Objective weights
    ph_q = weights[0] #100
    phi_e = weights[1] # 0.1
    phi_u = weights[2] #200
    objective = cp.Minimize(
        ph_q * cp.norm1(q_hat-q_hp) * delta_t + 
        phi_e * e ** 2 * delta_t +
        #phi_z * cp.norm2(z) * delta_t +
        phi_u * cp.norm2(u) * delta_t
        #phi_z * cp.maximum(0, -(q_hat + z)).sum() * delta_t
    )

    prob = cp.Problem(objective, constraints)
    _ = prob.solve(solver=cp.GUROBI, warm_start = False,  verbose=False)

    # Extract values
    R_a = 1 / a.value
    ws_val = w_s.value
    #ww_val = w_w.value
    w_val = w.value
    q_hat_val = q_hat.value

    # RMSE for dTi (u)
    rmse_dTi = np.sqrt(np.mean(u.value ** 2))

    # RMSE for q_hat vs q_hp
    rmse_q = np.sqrt(np.mean((q_hat.value - q_hp) ** 2))
    
    cost_q = np.linalg.norm(q_hat.value-q_hp,1) * delta_t
    cost_e = e.value ** 2 * delta_t 
    cost_u = np.linalg.norm(u.value,2) * delta_t

    return (R_a, ws_val, w_val, rmse_q, rmse_dTi, q_hat_val, Q_sc, cost_q, 
            cost_e, cost_u)

In [26]:
def read_q_exact_30min(df_train_q_exact: pd.DataFrame,
                       id_use,
                       t_start: pd.Timestamp,
                       t_end: pd.Timestamp) -> pd.DataFrame:
    """
    df_train_q_exact: wide DataFrame with
      index = Timestamp (30-min)
      columns = MultiIndex [Property_ID, variable]
        variable in {"Q_hp_total","Q_immersion","Q_dhw","Q_hp_sc","Q_total"}

    Returns a single-home DataFrame indexed by Timestamp with flat columns.
    Also creates:
      - Q_dhw_exact = Q_dhw
      - Q_spc_exact = Q_total - Q_dhw   (matches your earlier "q_tot outside DHW" intent)
    """
    # slice time first (fast) then pick this home's columns
    df_win = df_train_q_exact.loc[t_start:t_end]

    if not isinstance(df_win.columns, pd.MultiIndex):
        raise ValueError("Expected df_train_q_exact columns to be MultiIndex [Property_ID, variable].")

    if id_use not in df_win.columns.get_level_values(0):
        raise KeyError(f"id_use={id_use} not found in df_train_q_exact columns level 0 (Property_ID).")

    df_id = df_win.loc[:, (id_use, slice(None))].copy()
    df_id.columns = df_id.columns.droplevel(0)  # now columns are just the variables

    # Create the names your downstream code expects
    if "Q_dhw" in df_id.columns:
        df_id["Q_dhw_exact"] = df_id["Q_dhw"]

    # Your old code defined Q_spc_exact = q_tot when NOT DHW, else 0.
    # In the export script, Q_total is total (hp+immersion), and Q_dhw is Q_total during DHW else 0,
    # so: non-DHW total = Q_total - Q_dhw
    if "Q_total" in df_id.columns and "Q_dhw" in df_id.columns:
        df_id["Q_spc_exact"] = df_id["Q_total"] - df_id["Q_dhw"]

    return df_id


In [27]:
df_train_q_exact = pd.read_parquet(
    "retrieved_weather_data/q_streams_30min.parquet")

In [28]:
df_train_q_exact

Property_ID            EOH0005                                      EOH0021  \
variable            Q_hp_total Q_immersion Q_dhw Q_hp_sc Q_total Q_hp_total   
Timestamp                                                                     
2020-10-26 00:00:00        NaN         NaN   NaN     NaN     NaN        NaN   
2020-10-26 00:30:00        NaN         NaN   NaN     NaN     NaN        NaN   
2020-10-26 01:00:00        NaN         NaN   NaN     NaN     NaN        NaN   
2020-10-26 01:30:00        NaN         NaN   NaN     NaN     NaN        NaN   
2020-10-26 02:00:00        NaN         NaN   NaN     NaN     NaN        NaN   
...                        ...         ...   ...     ...     ...        ...   
2023-09-28 22:00:00        0.0         NaN   0.0     0.0     0.0        0.0   
2023-09-28 22:30:00        0.0         NaN   0.0     0.0     0.0        0.0   
2023-09-28 23:00:00        0.0         NaN   0.0     0.0     0.0        0.0   
2023-09-28 23:30:00        0.0         NaN   0.0     0.0     0.0        0.0   
2023-09-29 00:00:00        NaN         NaN   NaN     NaN     NaN        NaN   

Property_ID                                            ...    EOH3196  \
variable            Q_immersion Q_dhw Q_hp_sc Q_total  ... Q_hp_total   
Timestamp                                              ...              
2020-10-26 00:00:00         NaN   NaN     NaN     NaN  ...        NaN   
2020-10-26 00:30:00         NaN   NaN     NaN     NaN  ...        NaN   
2020-10-26 01:00:00         NaN   NaN     NaN     NaN  ...        NaN   
2020-10-26 01:30:00         NaN   NaN     NaN     NaN  ...        NaN   
2020-10-26 02:00:00         NaN   NaN     NaN     NaN  ...        NaN   
...                         ...   ...     ...     ...  ...        ...   
2023-09-28 22:00:00         0.0   0.0     0.0     0.0  ...        0.0   
2023-09-28 22:30:00         0.0   0.0     0.0     0.0  ...        0.0   
2023-09-28 23:00:00         0.0   0.0     0.0     0.0  ...        0.0   
2023-09-28 23:30:00         0.0   0.0     0.0     0.0  ...        0.0   
2023-09-29 00:00:00         NaN   NaN     NaN     NaN  ...        NaN   

Property_ID                                              EOH3204              \
variable            Q_immersion Q_dhw Q_hp_sc Q_total Q_hp_total Q_immersion   
Timestamp                                                                      
2020-10-26 00:00:00         NaN   NaN     NaN     NaN        NaN         NaN   
2020-10-26 00:30:00         NaN   NaN     NaN     NaN        NaN         NaN   
2020-10-26 01:00:00         NaN   NaN     NaN     NaN        NaN         NaN   
2020-10-26 01:30:00         NaN   NaN     NaN     NaN        NaN         NaN   
2020-10-26 02:00:00         NaN   NaN     NaN     NaN        NaN         NaN   
...                         ...   ...     ...     ...        ...         ...   
2023-09-28 22:00:00         0.0   0.0     0.0     0.0        NaN         NaN   
2023-09-28 22:30:00         0.0   0.0     0.0     0.0        NaN         NaN   
2023-09-28 23:00:00         0.0   0.0     0.0     0.0        NaN         NaN   
2023-09-28 23:30:00         0.0   0.0     0.0     0.0        NaN         NaN   
2023-09-29 00:00:00         NaN   NaN     NaN     NaN        NaN         NaN   

Property_ID                                
variable            Q_dhw Q_hp_sc Q_total  
Timestamp                                  
2020-10-26 00:00:00   NaN     NaN     NaN  
2020-10-26 00:30:00   NaN     NaN     NaN  
2020-10-26 01:00:00   NaN     NaN     NaN  
2020-10-26 01:30:00   NaN     NaN     NaN  
2020-10-26 02:00:00   NaN     NaN     NaN  
...                   ...     ...     ...  
2023-09-28 22:00:00   NaN     NaN     NaN  
2023-09-28 22:30:00   NaN     NaN     NaN  
2023-09-28 23:00:00   NaN     NaN     NaN  
2023-09-28 23:30:00   NaN     NaN     NaN  
2023-09-29 00:00:00   NaN     NaN     NaN  

[51265 rows x 1130 columns]

In [29]:
df_data = pd.read_parquet("training_data/initial_dataset.parquet")
df_train_detached = pd.read_parquet(
    "training_data/data_detached_with_weather.parquet")
type(df_data)
type(df_train_detached)
df_train_detached

,Property_ID,Timestamp,half-hour,Boiler_Energy_Output,Circulation_Pump_Energy_Consumed,Heat_Pump_Energy_Output,Whole_System_Energy_Consumed,External_Air_Temperature,Heat_Pump_Heating_Flow_Temperature,Heat_Pump_Return_Temperature,...,Postcode,Time,Temperature,FeelsLike,Humidity,Dew,Precipitation,SolarRadiation,SolarEnergy,Windspeed
0,EOH0279,2020-10-26 00:00:00,00:00:00,NaN,0.000,0.000,0.008,9.38,23.14,23.35,...,EH22,00:00:00,8.90,6.10,79.75,5.6,0.154,0.0,0.0,19.30
1,EOH1703,2020-10-26 00:00:00,00:00:00,NaN,0.013,0.242,0.078,7.08,25.04,24.04,...,NE15,00:00:00,7.00,4.30,81.71,4.1,0.170,0.0,0.0,14.30
2,EOH1703,2020-10-26 00:30:00,00:30:00,NaN,0.029,0.622,0.212,6.83,25.43,24.23,...,NE15,00:30:00,6.95,4.05,81.93,4.1,0.085,0.0,0.0,15.85
3,EOH0279,2020-10-26 00:30:00,00:30:00,NaN,0.000,0.000,0.019,9.33,22.54,22.80,...,EH22,00:30:00,8.85,6.30,79.94,5.6,0.077,0.0,0.0,17.00
4,EOH0279,2020-10-26 01:00:00,01:00:00,NaN,0.000,2.730,0.967,9.21,35.27,30.93,...,EH22,01:00:00,8.80,6.50,80.13,5.6,0.000,0.0,0.0,14.70
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
505054,EOH3154,2023-09-29 00:00:00,00:00:00,NaN,0.000,44801.768,14989.282,12.44,18.87,50.46,...,EH15,00:00:00,12.80,12.80,90.62,11.3,0.333,0.0,0.0,16.00
505055,EOH1637,2023-09-29 00:00:00,00:00:00,NaN,203.311,43996.381,13830.005,12.17,16.70,16.53,...,PH7,00:00:00,13.20,13.20,86.43,11.0,0.004,0.0,0.0,17.00
505056,EOH2329,2023-09-29 00:00:00,00:00:00,NaN,31.139,14128.100,5775.602,12.44,24.80,26.31,...,EH4,00:00:00,12.90,12.90,90.75,11.4,0.378,0.0,0.0,15.00
505057,EOH2675,2023-09-29 00:00:00,00:00:00,NaN,212.928,59675.471,19247.887,12.44,14.82,14.85,...,EH9,00:00:00,13.00,13.00,93.04,11.9,0.024,0.0,0.0,24.30


In [30]:
unique_ids = df_train_detached["Property_ID"].unique()
id_use = unique_ids[5]
df_house = pd.read_csv('training_data/home_characteristics.csv')

# List of columns to check for missingness
cols = [
    "Bedrooms", "Floor_Height", "Habitable_Rooms", "House_Age",
    "House_Form", "No_Storeys", "No_Underfloor",
    "Total_Floor_Area", "Wall_Type", "MCS_DHWAnnual","HP_Size_kW"
]

# calculate min capacity in home 
cp_air = 1.005  #kJ/kgK
rho_air = 1.225  # kg/m^3
kj_to_kWh = 1/3600
df_house["Volume"] = df_house["Total_Floor_Area"] * df_house["Floor_Height"]
df_house["Min Capacity"] = df_house["Volume"] * cp_air * rho_air * kj_to_kWh

df_home_values = df_house[df_house["Property_ID"].isin(unique_ids)]

summary = df_home_values[["Property_ID", "Min Capacity"]].drop_duplicates()
print(summary)

home_dict = df_home_values.set_index("Property_ID").to_dict(orient="index")


     Property_ID  Min Capacity
76       EOH3196      0.086941
318      EOH0413      0.094570
585      EOH2329      0.099645
736      EOH1637      0.088733
1149     EOH3154      0.116411
1156     EOH2675      0.124696
1569     EOH1485      0.115696
1692     EOH0546      0.088239
2066     EOH0279      0.138493
2381     EOH1703      0.074688


In [31]:
trained_params = pd.DataFrame(index=["Floor Area", "No_Storeys", "Wall_Type",
                                     "C", "R_a", "w_s", "w_w", "w",
                                     "Q_hat", "Q_sc", "rmse_dTi",
                                     "rmse_q_hp", "cost_q", "cost_e","cost_u"])

df_single = df_train_detached[df_train_detached["Property_ID"] == id_use].copy()
df_single = df_single.set_index('Timestamp')

#time range for indexing only
t_start = pd.to_datetime("2022 01 01 00:00:00")
t_mid_end = pd.to_datetime("2022 5 31 23:59:59")
t_mid_start = pd.to_datetime("2022 08 31 00:00:00")
t_end = pd.to_datetime("2022 12 31 23:59:59")
df_index = df_single[df_single.index >= t_start]
df_index = df_index[df_index.index <= t_end]
df_index = df_index[
    (df_index.index <= t_mid_end) | (df_index.index >= t_mid_start)]

# Create time index for q_hat arrays
time_index = df_index.index[:-1]
df_q_id = pd.DataFrame(index=time_index)


# ---- initialize and clean data ---------
wall = home_dict[id_use]["Wall_Type"]
storeys = home_dict[id_use]["No_Storeys"]
floor_area = home_dict[id_use]["Total_Floor_Area"]

df_single = df_train_detached[
    df_train_detached["Property_ID"] == id_use].copy()

#re-adjust Heat Pump Diff and add temp differences
df_single["Heat_Pump_Energy_Output_Diff"] = df_single[
    "Heat_Pump_Energy_Output"].diff()
df_single["Internal_Temperature_Diff"] = df_single[
    "Internal_Air_Temperature"].diff()
df_single["Internal_Ambient_Temperature_Diff"] = \
    (df_single["External_Air_Temperature"] -
     df_single["Internal_Air_Temperature"])

# 1. Drop columns with almost all missing data (e.g., more than 90% missing)
threshold = 0.90 * len(df_single)
df_single_cleaned = df_single.dropna(axis=1, thresh=threshold)
#print("Columns dropped due to high missing values:")
#print(df_single.columns.difference(df_single_cleaned.columns).tolist())

df_single = df_single_cleaned

#print("\nColumns remaining after dropping highly missing columns:")
#print(df_single.columns.tolist())

df_single = df_single.set_index('Timestamp')
df_single = df_single.sort_index()

# Apply interpolation with a limit of 4 (for 2 hours of half-hourly data)
numeric_cols = df_single.select_dtypes(include=['number']).columns
df_single_numeric_interpolated = df_single[numeric_cols].interpolate(
    method='time', limit=4, limit_direction='both')

df_single_interpolated = df_single.copy()
df_single_interpolated[numeric_cols] = df_single_numeric_interpolated

# After interpolation, drop rows that still contain NaN values (meaning they were missing for > 2 hours)
initial_rows = len(df_single_interpolated)
df_single_processed = df_single_interpolated.dropna()
rows_dropped_after_interpolation = initial_rows - len(df_single_processed)

#print(f"\nNumber of rows dropped after handling NA values (missing for >
# 2 hours): {rows_dropped_after_interpolation}")

# -----EXTRACT SUMMER DATA -------# 
df_heating_single = df_single_processed.copy()
#df_heating_single = [df_single_processed["External_Air_Temperature"]<=18]

# Time range
t_start = pd.to_datetime("2022-6-01 00:00:00")
t_end = pd.to_datetime("2022-08-30 23:59:00")
df_DHW = df_heating_single[
    (df_heating_single.index >= t_start) & (
                df_heating_single.index <= t_end)
    ].copy()

df_DHW.drop("Property_ID", axis=1, inplace=True)
df_DHW.drop("half-hour", axis=1, inplace=True)
df_DHW.drop("Date", axis=1, inplace=True)
df_DHW.drop("has_data", axis=1, inplace=True)

i = 24

df_numeric = df_DHW.select_dtypes(include='number')

# Resample and compute mean
df_resampled = df_numeric.resample(f'{i}h').mean()

df_DHW_only = df_resampled[
    df_resampled["Heat_Pump_Energy_Output_Diff"] <= 0.15]

DHW_sum = []
DHW_sum = df_DHW_only["Heat_Pump_Energy_Output_Diff"].sum() * 24

df_spc_only = df_resampled[
    df_resampled["Heat_Pump_Energy_Output_Diff"] > 0.15]
SPC_sum = df_spc_only["Heat_Pump_Energy_Output_Diff"].sum() * 24

# ------ Prepare Training Data ----------
df_heating_single = df_single_processed.copy()

#time range
t_start = pd.to_datetime("2022 01 01 00:00:00")
t_mid_end = pd.to_datetime("2022 5 31 23:59:59")
t_mid_start = pd.to_datetime("2022 08 31 00:00:00")
t_end = pd.to_datetime("2022 12 31 23:59:59")
df_heating_annual = df_heating_single[df_heating_single.index >= t_start]
df_heating_annual = df_heating_annual[df_heating_annual.index <= t_end]
df_heating_annual = df_heating_annual[
    (df_heating_annual.index <= t_mid_end) | (df_heating_annual
                                              .index >= t_mid_start)]

t_step = 30  # minutes
delta_t = t_step / 60

T_a = (df_heating_annual["External_Air_Temperature"].iloc[:-1].reset_index
       (drop=True).to_numpy())
T_i = (df_heating_annual["Internal_Air_Temperature"].iloc[:-1].reset_index
       (drop=True).to_numpy())
delta_T_a = (
    df_heating_annual["Internal_Ambient_Temperature_Diff"].iloc[:-1]
    .reset_index(drop=True).to_numpy())
delta_T_i = (df_heating_annual["Internal_Temperature_Diff"].iloc[1:]
             .reset_index(drop=True).to_numpy() / delta_t)
q_hp = (df_heating_annual["Heat_Pump_Energy_Output_Diff"].iloc[:-1]
        .reset_index(drop=True).to_numpy())
q_solar = (df_heating_annual["SolarRadiation"].iloc[:-1].reset_index
           (drop=True).to_numpy())
v_wind = (df_heating_annual["Windspeed"].iloc[:-1].reset_index(drop=True)
          .to_numpy())

N = len(delta_T_i)

Q_DHW_estimate = home_dict[id_use]["MCS_DHWAnnual"]
q_max = home_dict[id_use]["HP_Size_kW"]

# Create time index for in the loop q_hat arrays
time_index = df_heating_annual.index[:-1]
df_q_results = pd.DataFrame(index=time_index)

In [32]:
'''
t_start = pd.to_datetime("2022 01 01 00:00:00")
t_end   = pd.to_datetime("2022 01 30 23:59:00")

df_q_opt = df_q_results[(df_q_results.index >= t_start) & (df_q_results.index <= t_end)]
df_heating_val = df_heating_single[df_heating_single.index >= t_start]
df_heating_val = df_heating_val[df_heating_val.index <= t_end]

# Pull exact 30-min streams for this home
df_30min_val = read_q_exact_30min(df_train_q_exact, id_use, t_start, t_end)

# If you still want the total DHW over this window:
q_dhw_exact = 0.0
if "Q_dhw_exact" in df_30min_val.columns:
    q_dhw_exact = df_30min_val["Q_dhw_exact"].sum()
    '''


'\nt_start = pd.to_datetime("2022 01 01 00:00:00")\nt_end   = pd.to_datetime("2022 01 30 23:59:00")\n\ndf_q_opt = df_q_results[(df_q_results.index >= t_start) & (df_q_results.index <= t_end)]\ndf_heating_val = df_heating_single[df_heating_single.index >= t_start]\ndf_heating_val = df_heating_val[df_heating_val.index <= t_end]\n\n# Pull exact 30-min streams for this home\ndf_30min_val = read_q_exact_30min(df_train_q_exact, id_use, t_start, t_end)\n\n# If you still want the total DHW over this window:\nq_dhw_exact = 0.0\nif "Q_dhw_exact" in df_30min_val.columns:\n    q_dhw_exact = df_30min_val["Q_dhw_exact"].sum()\n    '

In [33]:
df_train_q_exact

Property_ID            EOH0005                                      EOH0021  \
variable            Q_hp_total Q_immersion Q_dhw Q_hp_sc Q_total Q_hp_total   
Timestamp                                                                     
2020-10-26 00:00:00        NaN         NaN   NaN     NaN     NaN        NaN   
2020-10-26 00:30:00        NaN         NaN   NaN     NaN     NaN        NaN   
2020-10-26 01:00:00        NaN         NaN   NaN     NaN     NaN        NaN   
2020-10-26 01:30:00        NaN         NaN   NaN     NaN     NaN        NaN   
2020-10-26 02:00:00        NaN         NaN   NaN     NaN     NaN        NaN   
...                        ...         ...   ...     ...     ...        ...   
2023-09-28 22:00:00        0.0         NaN   0.0     0.0     0.0        0.0   
2023-09-28 22:30:00        0.0         NaN   0.0     0.0     0.0        0.0   
2023-09-28 23:00:00        0.0         NaN   0.0     0.0     0.0        0.0   
2023-09-28 23:30:00        0.0         NaN   0.0     0.0     0.0        0.0   
2023-09-29 00:00:00        NaN         NaN   NaN     NaN     NaN        NaN   

Property_ID                                            ...    EOH3196  \
variable            Q_immersion Q_dhw Q_hp_sc Q_total  ... Q_hp_total   
Timestamp                                              ...              
2020-10-26 00:00:00         NaN   NaN     NaN     NaN  ...        NaN   
2020-10-26 00:30:00         NaN   NaN     NaN     NaN  ...        NaN   
2020-10-26 01:00:00         NaN   NaN     NaN     NaN  ...        NaN   
2020-10-26 01:30:00         NaN   NaN     NaN     NaN  ...        NaN   
2020-10-26 02:00:00         NaN   NaN     NaN     NaN  ...        NaN   
...                         ...   ...     ...     ...  ...        ...   
2023-09-28 22:00:00         0.0   0.0     0.0     0.0  ...        0.0   
2023-09-28 22:30:00         0.0   0.0     0.0     0.0  ...        0.0   
2023-09-28 23:00:00         0.0   0.0     0.0     0.0  ...        0.0   
2023-09-28 23:30:00         0.0   0.0     0.0     0.0  ...        0.0   
2023-09-29 00:00:00         NaN   NaN     NaN     NaN  ...        NaN   

Property_ID                                              EOH3204              \
variable            Q_immersion Q_dhw Q_hp_sc Q_total Q_hp_total Q_immersion   
Timestamp                                                                      
2020-10-26 00:00:00         NaN   NaN     NaN     NaN        NaN         NaN   
2020-10-26 00:30:00         NaN   NaN     NaN     NaN        NaN         NaN   
2020-10-26 01:00:00         NaN   NaN     NaN     NaN        NaN         NaN   
2020-10-26 01:30:00         NaN   NaN     NaN     NaN        NaN         NaN   
2020-10-26 02:00:00         NaN   NaN     NaN     NaN        NaN         NaN   
...                         ...   ...     ...     ...        ...         ...   
2023-09-28 22:00:00         0.0   0.0     0.0     0.0        NaN         NaN   
2023-09-28 22:30:00         0.0   0.0     0.0     0.0        NaN         NaN   
2023-09-28 23:00:00         0.0   0.0     0.0     0.0        NaN         NaN   
2023-09-28 23:30:00         0.0   0.0     0.0     0.0        NaN         NaN   
2023-09-29 00:00:00         NaN   NaN     NaN     NaN        NaN         NaN   

Property_ID                                
variable            Q_dhw Q_hp_sc Q_total  
Timestamp                                  
2020-10-26 00:00:00   NaN     NaN     NaN  
2020-10-26 00:30:00   NaN     NaN     NaN  
2020-10-26 01:00:00   NaN     NaN     NaN  
2020-10-26 01:30:00   NaN     NaN     NaN  
2020-10-26 02:00:00   NaN     NaN     NaN  
...                   ...     ...     ...  
2023-09-28 22:00:00   NaN     NaN     NaN  
2023-09-28 22:30:00   NaN     NaN     NaN  
2023-09-28 23:00:00   NaN     NaN     NaN  
2023-09-28 23:30:00   NaN     NaN     NaN  
2023-09-29 00:00:00   NaN     NaN     NaN  

[51265 rows x 1130 columns]

In [36]:
            
df_single = df_train_detached[df_train_detached["Property_ID"] == id_use].copy()

#re-adjust Heat Pump Diff and add temp differences
df_single["Heat_Pump_Energy_Output_Diff"] = df_single["Heat_Pump_Energy_Output"].diff()
df_single["Internal_Temperature_Diff"] = df_single["Internal_Air_Temperature"].diff()
df_single["Internal_Ambient_Temperature_Diff"] = \
    (df_single["External_Air_Temperature"] - 
     df_single["Internal_Air_Temperature"])
     


# 1. Drop columns with almost all missing data (e.g., more than 90% missing)
threshold = 0.90 * len(df_single)
df_single_cleaned = df_single.dropna(axis=1, thresh=threshold)

#print("Columns dropped due to high missing values:")
#print(df_single.columns.difference(df_single_cleaned.columns).tolist())

df_single = df_single_cleaned

#print("\nColumns remaining after dropping highly missing columns:")
#print(df_single.columns.tolist())

# 2. Handle missing values: Interpolate if missing for up to 2 hours (4 half-hour intervals), else drop rows
df_single = df_single.set_index('Timestamp')
df_single = df_single.sort_index()

# Apply interpolation with a limit of 4 (for 2 hours of half-hourly data)
numeric_cols = df_single.select_dtypes(include=['number']).columns
df_single_numeric_interpolated = df_single[numeric_cols].interpolate(method='time', limit=4, limit_direction='both')

df_single_interpolated = df_single.copy() 
df_single_interpolated[numeric_cols] = df_single_numeric_interpolated

# After interpolation, drop rows that still contain NaN values (meaning they were missing for > 2 hours)
initial_rows = len(df_single_interpolated)
df_single_processed = df_single_interpolated.dropna()
rows_dropped_after_interpolation = initial_rows - len(df_single_processed)

# -----EXTRACT SUMMER DATA -------# 
df_heating_single = df_single_processed

t_start = pd.to_datetime("2022 01 01 00:00:00")
t_end = pd.to_datetime("2022 01 30 23:59:00")
df_q_opt = df_q_results[(df_q_results.index>=t_start) & (df_q_results.index<=t_end)]
df_heating_val = df_heating_single[df_heating_single.index>=t_start]
df_heating_val = df_heating_val[df_heating_val.index<=t_end]

# --- Use precomputed 30-min exact streams (df_train_q_exact) instead of 2-min reconstruction

# Pull exact 30-min streams for this home and window
df_30min_val = read_q_exact_30min(df_train_q_exact, id_use, t_start, t_end)

# Keep these names alive so the rest of your script stays synchronized
q_dhw_exact = 0.0
q_dhw_exact_val = 0.0

# If available, compute DHW totals (optional but preserves your existing variables)
if "Q_dhw_exact" in df_30min_val.columns:
    q_dhw_exact = df_30min_val["Q_dhw_exact"].sum()
    q_dhw_exact_val = q_dhw_exact


T_a_val = (df_heating_val["External_Air_Temperature"].iloc[:-1].reset_index
       (drop=True).to_numpy()) 
T_i_val = (df_heating_val["Internal_Air_Temperature"].iloc[:-1].reset_index
       (drop=True).to_numpy()) 
delta_T_a_val = (df_heating_val["Internal_Ambient_Temperature_Diff"].iloc[:-1]
             .reset_index(drop=True).to_numpy()) 
delta_T_i_val = (df_heating_val["Internal_Temperature_Diff"].iloc[1:]
             .reset_index(drop=True).to_numpy() / delta_t) 
q_hp_val = (df_heating_val["Heat_Pump_Energy_Output_Diff"].iloc[:-1]
        .reset_index(drop=True).to_numpy()) 
q_solar_val = (df_heating_val["SolarRadiation"].iloc[:-1].reset_index
           (drop=True).to_numpy()) 
v_wind_val = (df_heating_val["Windspeed"].iloc[:-1].reset_index(drop=True)
          .to_numpy()) 
q_real_val = df_30min_val["Q_spc_exact"].iloc[:-1].reset_index(drop=True).to_numpy()

# training time base (matches q_hp, delta_T_a slices iloc[:-1])
train_time = df_heating_annual.index[:-1]

# Build validation indices directly from the training timeline (works with duplicate timestamps)
tmask_val = (train_time >= t_start) & (train_time <= t_end)
val_idx = np.flatnonzero(tmask_val)

# You used .iloc[:-1] after loc[t_start:t_end] for validation arrays, so mimic that:
val_idx = val_idx[:-1]

# Hard alignment check
assert len(val_idx) == len(delta_T_i_val), (len(val_idx), len(delta_T_i_val))




In [37]:
del df_single_cleaned, df_single_interpolated, df_single_processed, (
    df_heating_single), df_single, df_30min_val, df_data, df_train_detached, df_DHW, (
    df_DHW_only), df_home_values, df_single_numeric_interpolated, df_house

In [ ]:
trained_params_val = pd.DataFrame(index=["Floor Area", "No_Storeys", "Wall_Type",
                                     "C", "R_a", "w_s", "w_w", "w",
                                     "Q_hat", "Q_sc", "rmse_dTi_train",
                                     "rmse_q_hp_train", "rmse_dTi_val",
                                     "rmse_q_hp_val","cost_q", "cost_e",
                                         "cost_u"])
phi_q = np.arange(0, 500, 50)
phi_q[0] = 1
phi_e = 1/phi_q 
phi_u = phi_q

patience_k = 3          # how many non-improving k's before stopping
min_improve = 0.01      # relative improvement threshold (1%)
alpha_T = 1.0           # scalarization weight (optional)
alpha_q = 1.0

# Sentinel k values to guard against "good at high k"
k_sentinels = [phi_u[0], phi_u[-1]]

qhat_store = {}



N = len(delta_T_i)
#inner = None # InnerSolver(N=N, q_max=q_max, delta_t=delta_t)
inner = InnerSolver(N=N, q_max=q_max, delta_t=delta_t)
log_mem("created InnerSolver once")

for i in phi_q:
    for j in phi_e:
        RSS0 = rss_gb()
        #inner = reset_inner_solver(inner, N=N, q_max=q_max, delta_t=delta_t,
        #                           tag=f"(phi_q={i}, phi_e={j})")
        # Track best performance for this (i, j)
        best_score_ij = np.inf
        bad_k = 0

        # -----------------------------
        # 1) Sentinel evaluations
        # -----------------------------
        for k in k_sentinels:
            weights = [i, j, k]
            print("Trying in k:", weights)
            # Grid search and store results
            C_values = np.arange(1.5, 20, 1)
            
            best_obj = (np.inf, np.inf)   # (rmse_dTi, rmse_q)
            best_row = None               # dict for best C only
            best_q_hat_val = None
            
            

            for C in C_values:
                log_mem("before solve_inner")
                '''
                R_a, ws, w, rmse_q, rmse_dTi, q_hat_val, Q_sc, cost_q, cost_e, cost_u = (
                    solve_inner(C, q_hp, delta_T_i, delta_T_a, q_solar, SPC_sum, DHW_sum,
                                Q_DHW_estimate, q_max, delta_t, weights))
                '''
                
                Q_sc = np.sum(q_hp) - Q_DHW_estimate + DHW_sum  # same as your current
                out, inner = safe_solve(
                    inner,
                    C=C, q_hp=q_hp, delta_T_i=delta_T_i, delta_T_a=delta_T_a,
                    q_solar=q_solar, Q_sc=Q_sc, weights=weights,
                    solver=cp.GUROBI,
                    max_retries=1,
                    tag=f"(sentinel phi_q={i},phi_e={j},phi_u={k})"
                )

                if out is None:
                    print(f"[SKIP C] weights={weights} C={C} (solve failed)")
                    continue

                R_a, ws, w, rmse_q, rmse_dTi, q_hat_val, cost_q, cost_e, cost_u = out

                if mem_alarm(tag=f"sentinel phi_q={i},phi_e={j},phi_u={k}", growth_gb=1.0):
                    inner = reset_inner_solver(
                        inner, N=N, q_max=q_max, delta_t=delta_t, tag="mem alarm"
                    )
                    RSS0 = rss_gb()   # reset baseline after rebuild

                '''
                R_a, ws, w, rmse_q, rmse_dTi, q_hat_val, cost_q, cost_e, cost_u = inner.solve(
                    C=C,
                    q_hp=q_hp,
                    delta_T_i=delta_T_i,
                    delta_T_a=delta_T_a,
                    q_solar=q_solar,
                    Q_sc=Q_sc,
                    weights=weights,
                    solver=cp.GUROBI  
                )
                '''
                
                #Save each q_hat under C value for iteration
                #df_q_results[f"C_{C:.1f}"] = q_hat_val
                log_mem("after solve_inner")
            
                #print(f"C={C:.2f} | R_a={R_a:.4f} | w_s={ws:.4f} | w={w:.4f} | "
                #      f"rmse_q={rmse_q:.4f} | rmse_dTi={rmse_dTi:.4f}")
                obj = (rmse_dTi, rmse_q)
                if obj < best_obj:
                    best_obj = obj
                    best_row = {
                        "C": float(C),
                        "R_a": float(R_a),
                        "w_s": float(ws),
                        "w": float(w),  # fixes array(0.65)
                        "Q_hat": float(np.sum(q_hat_val)),
                        "Q_sc": float(Q_sc),
                        "rmse_q": float(rmse_q),
                        "rmse_dTi": float(rmse_dTi),
                        "cost_q": float(cost_q),
                        "cost_e": float(cost_e),
                        "cost_u": float(cost_u),
                    }
                    # only keep q_hat for the best candidate (no need to copy every time)
                    best_q_hat_val = q_hat_val
            
            # Convert to DataFrame
            top_result = best_row
            print(top_result)
            #df_q_id[id_use] = best_q_hat_val
            
            '''
            print("Optimal C_a:", top_result["C"])
            print("Optimal R_a:", top_result["R_a"])
            print("w_s (solar gain):", top_result["w_s"])
            print("w_w (wind effect):", 'NA')
            print("w (bias):", top_result["w"])
            print("Q_hat total:", top_result["Q_hat"])
            print("Q_sc:", top_result["Q_sc"])
            print("RMSE dTi:", top_result["rmse_dTi"])
            print("RMSE q_hat:", top_result["rmse_q"])
            '''
            
            trained_params[id_use] = [floor_area, storeys, wall, top_result["C"],
                                      top_result["R_a"], top_result["w_s"], "NA",
                                      top_result["w"], top_result["Q_hat"],
                                      top_result["Q_sc"], top_result["rmse_dTi"],
                                      top_result["rmse_q"], top_result["cost_q"],
                                      top_result["cost_e"], top_result["cost_u"]]
            
            df_id = trained_params[id_use].copy()
            C = df_id.C
            R_a = df_id.R_a
            w_s= df_id.w_s
            w = df_id.w
            
            #print(f"\nNumber of rows dropped after handling NA values (missing for >
            # 2 hours): {rows_dropped_after_interpolation}")
            

            # Build a temporary time-indexed series for q_hat and slice it (no df_q_id storage needed)
            #q_opt_series = pd.Series(
            #    best_q_hat_val,
            #    index=df_q_results.index[:len(best_q_hat_val)]  # aligns
            #    q_hat to your q_results timestamps
            #)
            
            #q_opt = q_opt_series.loc[t_start:t_end].iloc[:-1].to_numpy()

            q_opt = best_q_hat_val[val_idx]

            q_hat_sim = (delta_T_i_val * C -  delta_T_a_val / R_a * delta_t  - w_s *  
                    q_solar_val  - w * np.ones(len(T_a_val), )* delta_t )
            
            delta_T_i_sim = (delta_T_a_val / R_a *delta_t + q_opt + w_s* q_solar_val + w
                         * np
                         .ones(len(T_a_val), )* delta_t) / C
            
            # Preallocate
            T_i_sim = np.zeros_like(delta_T_i_sim)
            
            # Time index aligned with delta_T_i_sim
            time_idx = df_heating_val.index[1:]
            
            # Initial condition (first sample)
            T_i_prev = T_i_val[0]
            
            for i_1, t in enumerate(time_idx):
            
                # If midnight, reinitialize from measurement
                if t.hour == 0 and t.minute == 0:
                    T_i_prev = df_heating_val["Internal_Air_Temperature"].loc[t]
            
                # State update
                T_i_sim[i_1] = T_i_prev + delta_T_i_sim[i_1] * delta_t
                T_i_prev = T_i_sim[i_1]
            
            e_Ti = T_i_sim - df_heating_val["Internal_Air_Temperature"].iloc[1:]    
            e_q = q_hat_sim - q_real_val
            rmse_q_val = np.sqrt(np.mean(e_q**2))
            rmse_Ti = np.sqrt(np.mean(e_Ti**2))
            print(f"RMSE of q_hp: {rmse_q_val}")
            print(f"RMSE of Ti: {rmse_Ti}")
            
            trained_params_val[(i, j, k)] = [
                floor_area, storeys, wall, top_result["C"],
                top_result["R_a"], top_result["w_s"], "NA",
                top_result["w"], top_result["Q_hat"],
                top_result["Q_sc"], top_result["rmse_dTi"],
                top_result["rmse_q"], rmse_Ti,
                rmse_q_val, top_result["cost_q"],
                top_result["cost_e"], top_result["cost_u"]
            ]
            
            
            log_mem("after validation sim")
            
            # ------ Break early -------------
            score = alpha_T * rmse_Ti + alpha_q * rmse_q_val

        for k in phi_u:
            if k in k_sentinels:
                continue  # already evaluated

            weights = [i, j, k]
            print("Trying in phi_u:", weights)
            # Grid search and store results
            C_values = np.arange(1.5, 20, 1)

            best_obj = (np.inf, np.inf)   # (rmse_dTi, rmse_q)
            best_row = None               # dict for best C only
            best_q_hat_val = None

            #inner = reset_inner_solver(
            #    inner, N=N, q_max=q_max, delta_t=delta_t,
            #    tag=f"before phi_u sweep (phi_q={i}, phi_e={j})"
            #)


            for C in C_values:
                log_mem("before solve_inner")
                '''
                R_a, ws, w, rmse_q, rmse_dTi, q_hat_val, Q_sc, cost_q, cost_e, cost_u = (
                    solve_inner(C, q_hp, delta_T_i, delta_T_a, q_solar, SPC_sum, DHW_sum,
                                Q_DHW_estimate, q_max, delta_t, weights))
                '''

                Q_sc = np.sum(q_hp) - Q_DHW_estimate + DHW_sum  # same as your current
                out, inner = safe_solve(
                    inner,
                    C=C, q_hp=q_hp, delta_T_i=delta_T_i, delta_T_a=delta_T_a,
                    q_solar=q_solar, Q_sc=Q_sc, weights=weights,
                    solver=cp.GUROBI,
                    max_retries=1,
                    tag=f"(phi_q={i},phi_e={j},phi_u={k},C={C})"
                )

                if out is None:
                    print(f"[SKIP C] weights={weights} C={C} (solve failed)")
                    continue

                R_a, ws, w, rmse_q, rmse_dTi, q_hat_val, cost_q, cost_e, cost_u = out
                log_mem("after solve_inner")

                if mem_alarm(tag=f"sentinel phi_q={i},phi_e={j},phi_u={k}", growth_gb=1.0):
                    inner = reset_inner_solver(
                        inner, N=N, q_max=q_max, delta_t=delta_t, tag="mem alarm"
                    )
                    RSS0 = rss_gb()   # reset baseline after rebuild


                # extra guard (cheap)
                if (q_hat_val is None) or (not np.all(np.isfinite(q_hat_val))):
                    print(f"[SKIP C] weights={weights} C={C} (bad q_hat)")
                    continue


                obj = (rmse_dTi, rmse_q)
                if obj < best_obj:
                    best_obj = obj
                    best_row = {
                        "C": float(C),
                        "R_a": float(R_a),
                        "w_s": float(ws),
                        "w": float(w),  # fixes array(0.65)
                        "Q_hat": float(np.sum(q_hat_val)),
                        "Q_sc": float(Q_sc),
                        "rmse_q": float(rmse_q),
                        "rmse_dTi": float(rmse_dTi),
                        "cost_q": float(cost_q),
                        "cost_e": float(cost_e),
                        "cost_u": float(cost_u),
                    }
                    # only keep q_hat for the best candidate (no need to copy every time)
                    best_q_hat_val = q_hat_val

            if best_row is None or best_q_hat_val is None:
                print(f"[SKIP WEIGHT] weights={weights} (no feasible C solved)")
                continue

            # Convert to DataFrame
            top_result = best_row
            print(top_result)
            #df_q_id[id_use] = best_q_hat_val

            '''
            print("Optimal C_a:", top_result["C"])
            print("Optimal R_a:", top_result["R_a"])
            print("w_s (solar gain):", top_result["w_s"])
            print("w_w (wind effect):", 'NA')
            print("w (bias):", top_result["w"])
            print("Q_hat total:", top_result["Q_hat"])
            print("Q_sc:", top_result["Q_sc"])
            print("RMSE dTi:", top_result["rmse_dTi"])
            print("RMSE q_hat:", top_result["rmse_q"])
            '''

            C = top_result.C
            R_a = top_result.R_a
            w_s= top_result.w_s
            w = top_result.w



            # Build a temporary time-indexed series for q_hat and slice it (no df_q_id storage needed)
            '''
            q_opt_series = pd.Series(
                best_q_hat_val,
                index=df_q_results.index[:len(best_q_hat_val)]  # aligns q_hat to your q_results timestamps
            )

            q_opt = q_opt_series.loc[t_start:t_end].iloc[:-1].to_numpy()
            '''
            q_opt = best_q_hat_val[val_idx]
            q_hat_sim = (delta_T_i_val * C -  delta_T_a_val / R_a * delta_t  - w_s *
                    q_solar_val  - w * np.ones(len(T_a_val), )* delta_t )

            delta_T_i_sim = (delta_T_a_val / R_a *delta_t + q_opt + w_s* q_solar_val + w
                         * np
                         .ones(len(T_a_val), )* delta_t) / C

            # Preallocate
            T_i_sim = np.zeros_like(delta_T_i_sim)

            # Time index aligned with delta_T_i_sim
            time_idx = df_heating_val.index[1:]

            # Initial condition (first sample)
            T_i_prev = T_i_val[0]

            for i_1, t in enumerate(time_idx):

                # If midnight, reinitialize from measurement
                if t.hour == 0 and t.minute == 0:
                    T_i_prev = df_heating_val["Internal_Air_Temperature"].loc[t]

                # State update
                T_i_sim[i_1] = T_i_prev + delta_T_i_sim[i_1] * delta_t
                T_i_prev = T_i_sim[i_1]

            e_Ti = T_i_sim - df_heating_val["Internal_Air_Temperature"].iloc[1:]
            e_q = q_hat_sim - q_real_val
            rmse_q_val = np.sqrt(np.mean(e_q**2))
            rmse_Ti = np.sqrt(np.mean(e_Ti**2))
            print(f"RMSE of q_hp: {rmse_q_val}")
            print(f"RMSE of Ti: {rmse_Ti}")

            trained_params_val[(i, j, k)] = [
                floor_area, storeys, wall, top_result["C"],
                top_result["R_a"], top_result["w_s"], "NA",
                top_result["w"], top_result["Q_hat"],
                top_result["Q_sc"], top_result["rmse_dTi"],
                top_result["rmse_q"], rmse_Ti,
                rmse_q_val, top_result["cost_q"],
                top_result["cost_e"], top_result["cost_u"]
            ]

            #inner = reset_inner_solver(
            #    inner, N=N, q_max=q_max, delta_t=delta_t,
            #    tag=f"before phi_u sweep (phi_q={i}, phi_e={j})"
            #    )
            # ------- Break Early ----------
            score = alpha_T * rmse_Ti + alpha_q * rmse_q_val

            # Improvement check
            if score < best_score_ij * (1.0 - min_improve):
                best_score_ij = score
                bad_k = 0
            else:
                bad_k += 1

            # Early stop along k
            if bad_k >= patience_k:
                print(
                    f"Early-stop k for (phi_q={i}, phi_e={j}) "
                    f"after k={k}: no further improvement."
                )
                break


                
gc.collect()
log_mem("after gc.collect")
qhat_store[(i, j, k)] = best_q_hat_val.astype(np.float32, copy=False)



        
        
    

DCP: True DPP: False
[mem] created InnerSolver once       RSS=0.84 GB
Trying in k: [np.int64(1), np.float64(1.0), np.int64(1)]
[mem] before solve_inner             RSS=0.84 GB
Set parameter Username
Academic license - for non-commercial use only - expires 2026-06-05


/Users/levipremer/PycharmProjects/uk_data_electric_heating/.venv/lib/python3.12/site-packages/cvxpy/reductions/solvers/solving_chain.py:241: UserWarning: You are solving a parameterized problem that is not DPP. Because the problem is not DPP, subsequent solves will not be faster than the first one. For more information, see the documentation on Disciplined Parametrized Programming, at https://www.cvxpy.org/tutorial/dpp/index.html
  warnings.warn(DPP_ERROR_MSG)


[mem] after solve_inner              RSS=1.02 GB
[mem] before solve_inner             RSS=1.02 GB
[mem] after solve_inner              RSS=0.99 GB
[mem] before solve_inner             RSS=0.99 GB
[mem] after solve_inner              RSS=0.99 GB
[mem] before solve_inner             RSS=0.99 GB
[mem] after solve_inner              RSS=1.06 GB
[mem] before solve_inner             RSS=1.06 GB
[mem] after solve_inner              RSS=1.09 GB
[mem] before solve_inner             RSS=1.09 GB
[mem] after solve_inner              RSS=1.10 GB
[mem] before solve_inner             RSS=1.10 GB
[mem] after solve_inner              RSS=1.15 GB
[mem] before solve_inner             RSS=1.15 GB
[mem] after solve_inner              RSS=1.17 GB
[mem] before solve_inner             RSS=1.17 GB
[mem] after solve_inner              RSS=1.17 GB
[mem] before solve_inner             RSS=1.17 GB
[mem] after solve_inner              RSS=1.17 GB
[mem] before solve_inner             RSS=1.17 GB
[mem] after solve_in

In [ ]:
df_q_id

In [ ]:
cols = ["Floor Area", "No_Storeys", "Wall_Type",
        "C", "R_a", "w_s", "w_w", "w",
        "Q_hat", "Q_sc", "rmse_dTi_train",
        "rmse_q_hp_train", "rmse_dTi_val",
        "rmse_q_hp_val", "cost_q", "cost_e", "cost_u"]

df_trained_val = (
    pd.DataFrame.from_dict(trained_params_val, orient="index", columns=cols)
)

# Name the index nicely
df_trained_val.index = pd.MultiIndex.from_tuples(
    df_trained_val.index,
    names=["phi_q", "phi_e", "phi_u"]
)


In [ ]:
plt.figure()
plt.scatter(df_trained_val["rmse_dTi_val"],
            df_trained_val["rmse_q_hp_val"])
plt.grid(True)
plt.xlabel("Validation RMSE of Ti")
plt.ylabel("Validation RMSE of q")
plt.title("Pareto front over weight sets")
plt.show()


In [ ]:
trained_params

In [ ]:
print(Q_DHW_estimate)
print(DHW_sum)
print(q_hp.sum())

print(Q_sc)

In [ ]:
# ------- Look at heat supply performance in validation data ------

df_id = trained_params[id_use].copy()
C = df_id.C
R_a = df_id.R_a
w_s= df_id.w_s
w = df_id.w


df_single = df_train_detached[df_train_detached["Property_ID"] == id_use].copy()

#re-adjust Heat Pump Diff and add temp differences
df_single["Heat_Pump_Energy_Output_Diff"] = df_single["Heat_Pump_Energy_Output"].diff()
df_single["Internal_Temperature_Diff"] = df_single["Internal_Air_Temperature"].diff()
df_single["Internal_Ambient_Temperature_Diff"] = \
    (df_single["External_Air_Temperature"] - 
     df_single["Internal_Air_Temperature"])
     


# 1. Drop columns with almost all missing data (e.g., more than 90% missing)
threshold = 0.90 * len(df_single)
df_single_cleaned = df_single.dropna(axis=1, thresh=threshold)
#print("Columns dropped due to high missing values:")
#print(df_single.columns.difference(df_single_cleaned.columns).tolist())

df_single = df_single_cleaned

#print("\nColumns remaining after dropping highly missing columns:")
#print(df_single.columns.tolist())

# 2. Handle missing values: Interpolate if missing for up to 2 hours (4 half-hour intervals), else drop rows
df_single = df_single.set_index('Timestamp')
df_single = df_single.sort_index()

# Apply interpolation with a limit of 4 (for 2 hours of half-hourly data)
numeric_cols = df_single.select_dtypes(include=['number']).columns
df_single_numeric_interpolated = df_single[numeric_cols].interpolate(method='time', limit=4, limit_direction='both')

df_single_interpolated = df_single.copy() 
df_single_interpolated[numeric_cols] = df_single_numeric_interpolated

# After interpolation, drop rows that still contain NaN values (meaning they were missing for > 2 hours)
initial_rows = len(df_single_interpolated)
df_single_processed = df_single_interpolated.dropna()
rows_dropped_after_interpolation = initial_rows - len(df_single_processed)

#print(f"\nNumber of rows dropped after handling NA values (missing for >
# 2 hours): {rows_dropped_after_interpolation}")

# -----EXTRACT SUMMER DATA -------# 
df_heating_single = df_single_processed.copy()

t_start = pd.to_datetime("2023 01 01 00:00:00")
t_end = pd.to_datetime("2023 01 05 23:59:00")
df_q_opt = df_q_results[(df_q_results.index>=t_start) & (df_q_results.index<=t_end)]
df_heating_val = df_heating_single[df_heating_single.index>=t_start]
df_heating_val = df_heating_val[df_heating_val.index<=t_end]



T_a_val = (df_heating_val["External_Air_Temperature"].iloc[:-1].reset_index
       (drop=True).to_numpy()) 
T_i_val = (df_heating_val["Internal_Air_Temperature"].iloc[:-1].reset_index
       (drop=True).to_numpy()) 
delta_T_a_val = (df_heating_val["Internal_Ambient_Temperature_Diff"].iloc[:-1]
             .reset_index(drop=True).to_numpy()) 
delta_T_i_val = (df_heating_val["Internal_Temperature_Diff"].iloc[1:]
             .reset_index(drop=True).to_numpy() / delta_t) 
q_hp_val = (df_heating_val["Heat_Pump_Energy_Output_Diff"].iloc[:-1]
        .reset_index(drop=True).to_numpy()) 
q_solar_val = (df_heating_val["SolarRadiation"].iloc[:-1].reset_index
           (drop=True).to_numpy()) 
v_wind_val = (df_heating_val["Windspeed"].iloc[:-1].reset_index(drop=True)
          .to_numpy()) 



q_hat_sim = (delta_T_i_val * C -  delta_T_a_val / R_a * delta_t  - w_s *  
        q_solar_val  - w * np.ones(len(T_a_val), )* delta_t )

e_q = q_hat_sim - q_hp_val
rmse_q_val = np.sqrt(np.mean(e_q**2))
print(f"RMSE of q_hp: {rmse_q_val}")

plt.plot(df_heating_val.index[1:],q_hat_sim, label = "Sim")
plt.plot(df_heating_val.index[1:], q_hp_val, label = "Measured")
plt.legend()
plt.xlabel("Time")
plt.ylabel("kWh")
plt.title(f"{id_use}")
plt.show()

In [ ]:
# -----Look at temperature tracking performance in the training --------------

df_id = trained_params[id_use].copy()
C = df_id.C
R_a = df_id.R_a
w_s= df_id.w_s
w = df_id.w

df_single = df_train_detached[df_train_detached["Property_ID"] == id_use].copy()

#re-adjust Heat Pump Diff and add temp differences
df_single["Heat_Pump_Energy_Output_Diff"] = df_single["Heat_Pump_Energy_Output"].diff()
df_single["Internal_Temperature_Diff"] = df_single["Internal_Air_Temperature"].diff()
df_single["Internal_Ambient_Temperature_Diff"] = \
    (df_single["External_Air_Temperature"] - 
     df_single["Internal_Air_Temperature"])
     


# 1. Drop columns with almost all missing data (e.g., more than 90% missing)
threshold = 0.90 * len(df_single)
df_single_cleaned = df_single.dropna(axis=1, thresh=threshold)
#print("Columns dropped due to high missing values:")
#print(df_single.columns.difference(df_single_cleaned.columns).tolist())

df_single = df_single_cleaned

#print("\nColumns remaining after dropping highly missing columns:")
#print(df_single.columns.tolist())

# 2. Handle missing values: Interpolate if missing for up to 2 hours (4 half-hour intervals), else drop rows
df_single = df_single.set_index('Timestamp')
df_single = df_single.sort_index()

# Apply interpolation with a limit of 4 (for 2 hours of half-hourly data)
numeric_cols = df_single.select_dtypes(include=['number']).columns
df_single_numeric_interpolated = df_single[numeric_cols].interpolate(method='time', limit=4, limit_direction='both')

df_single_interpolated = df_single.copy() 
df_single_interpolated[numeric_cols] = df_single_numeric_interpolated

# After interpolation, drop rows that still contain NaN values (meaning they were missing for > 2 hours)
initial_rows = len(df_single_interpolated)
df_single_processed = df_single_interpolated.dropna()
rows_dropped_after_interpolation = initial_rows - len(df_single_processed)

#print(f"\nNumber of rows dropped after handling NA values (missing for >
# 2 hours): {rows_dropped_after_interpolation}")

# -----EXTRACT SUMMER DATA -------# 
df_heating_single = df_single_processed.copy()

# Validate results
t_start = pd.to_datetime("2022 01 01 00:00:00")
t_end = pd.to_datetime("2022 01 02 23:59:00")
df_q_id_use = df_q_id[(df_q_id.index>=t_start) & (df_q_id.index<=t_end)]
df_heating_val = df_heating_single[df_heating_single.index>=t_start]
df_heating_val = df_heating_val[df_heating_val.index<=t_end]



T_a_val = (df_heating_val["External_Air_Temperature"].iloc[:-1].reset_index
       (drop=True).to_numpy()) 
T_i_val = (df_heating_val["Internal_Air_Temperature"].iloc[:-1].reset_index
       (drop=True).to_numpy()) 
delta_T_a_val = (df_heating_val["Internal_Ambient_Temperature_Diff"].iloc[:-1]
             .reset_index(drop=True).to_numpy()) 
delta_T_i_val = (df_heating_val["Internal_Temperature_Diff"].iloc[1:]
             .reset_index(drop=True).to_numpy() / delta_t) 
q_hp_val = (df_heating_val["Heat_Pump_Energy_Output_Diff"].iloc[:-1]
        .reset_index(drop=True).to_numpy()) 
q_opt = df_q_id_use[id_use].iloc[:-1].reset_index(drop=True).to_numpy()
q_solar_val = (df_heating_val["SolarRadiation"].iloc[:-1].reset_index
           (drop=True).to_numpy()) 
v_wind_val = (df_heating_val["Windspeed"].iloc[:-1].reset_index(drop=True)
          .to_numpy()) 


delta_T_i_sim = (delta_T_a_val / R_a *delta_t + q_opt + w_s* q_solar_val + w
                 * np
                 .ones(len(T_a_val), )* delta_t) / C

T_i_sim = np.zeros_like(T_i_val)
T_i_prev = T_i_val[0]
for i in range(len(delta_T_i_sim)):
    T_i_sim[i] = T_i_prev + delta_T_i_sim[i]*delta_t
    T_i_prev = T_i_sim[i]


plt.plot(df_heating_val.index[1:],q_opt, label = "Optimized q")
plt.plot(df_heating_val.index[1:],q_hp_val, label = "Measured q")

plt.legend()
plt.xlabel("Time")
plt.ylabel("kWh")
plt.title(f"{id_use}")
plt.show()

plt.plot(df_heating_val.index[1:],delta_T_i_sim, label = "Sim")
plt.plot(df_heating_val.index[1:], delta_T_i_val, label = "Measured")
plt.legend()
plt.xlabel("Time")
plt.ylabel("Delta T")
plt.title(f"{id_use}")
plt.show()

plt.plot(df_heating_val.index[1:],T_i_sim, label = "Sim")
plt.plot(df_heating_val.index[1:], df_heating_val["Internal_Air_Temperature"]
         .iloc[1:].reset_index
       (drop=True).to_numpy(), label = "Measured")
plt.legend()
plt.xlabel("Time")
plt.ylabel("Celsius")
plt.title(f"{id_use}")
plt.show()